In [1]:
from dotenv import load_dotenv
load_dotenv()

True

In [2]:
from openai import OpenAI
openai_client = OpenAI()

In [26]:
from ingest import load_faq_data, build_index

documents = load_faq_data()
index = build_index(documents)

In [27]:
def search(query):
    boost_dict = {'question':3.0, 'section':0.5}
    filter_dict = {'course':'llm-zoomcamp'}

    return index.search(
        query,
        num_results=5,
        boost_dict=boost_dict,
        filter_dict=filter_dict
    )

In [28]:
search_tool = {
    "type": "function",
    "name": "search",
    "description": "Search the FAQ database for entries matching the given query.",
    "parameters": {
        "type": "object",
        "properties": {
            "query": {
                "type": "string",
                "description":"Search query text to look up in the course FAQ."
            }
        },
        "required": ["query"],
        "additionalProperties":False
    }
}

In [51]:
messages = [
    {'role': 'user', 'content': 'I just discovered the course. Can I joint it?'}
]

response = openai_client.responses.create(
    model='gpt-5.4-mini',
    input=messages,
    tools=[search_tool]
)

response.output

[ResponseFunctionToolCall(arguments='{"query":"join course enroll join discovered course can I join"}', call_id='call_okSKxREOhR02fKIcKjYt1r9Y', name='search', type='function_call', id='fc_0295230bc459387d006a2ef0940cac81989856a71330c7cef5', namespace=None, status='completed')]

In [52]:
call = response.output[0]

In [53]:
call.arguments

'{"query":"join course enroll join discovered course can I join"}'

In [54]:
import json

args = json.loads(call.arguments)

search_results = search(**args)

result_json = json.dumps(search_results, indent=2)


In [55]:
messages.extend(response.output)

In [56]:
function_call_output = {
    "type": "function_call_output",
    "call_id": call.call_id,
    "output": result_json
}

In [57]:
messages.append(function_call_output)

In [58]:
messages

[{'role': 'user', 'content': 'I just discovered the course. Can I joint it?'},
 ResponseFunctionToolCall(arguments='{"query":"join course enroll join discovered course can I join"}', call_id='call_okSKxREOhR02fKIcKjYt1r9Y', name='search', type='function_call', id='fc_0295230bc459387d006a2ef0940cac81989856a71330c7cef5', namespace=None, status='completed'),
 {'type': 'function_call_output',
  'call_id': 'call_okSKxREOhR02fKIcKjYt1r9Y',
  'output': '[\n  {\n    "id": "74eb249bbf",\n    "course": "llm-zoomcamp",\n    "section": "General Course-Related Questions",\n    "question": "I just discovered the course. Can I still join?",\n    "answer": "Yes, but if you want to receive a certificate, you need to submit your project while we\\u2019re still accepting submissions."\n  },\n  {\n    "id": "69d122f12e",\n    "course": "llm-zoomcamp",\n    "section": "General Course-Related Questions",\n    "question": "Certificate: Can I follow the course in a self-paced mode and get a certificate?",\n  

In [60]:
response = openai_client.responses.create(
    model='gpt-5.4-mini',
    input=messages,
    tools=[search_tool]
)

In [61]:
print(response.output_text)

Yes, you can still join the course. If you want to receive a certificate, you need to submit your project while submissions are still open.


In [63]:
usage = response.usage
usage.input_tokens, usage.output_tokens

(650, 33)

In [64]:
def calculate_gpt54mini_price(input_tokens, output_tokens):
    INPUT_PRICE_PER_MILLION = 0.15
    OUTPUT_PRICE_PER_MILLION = 0.60

    input_cost = (input_tokens / 1_000_000) * INPUT_PRICE_PER_MILLION
    output_cost = (output_tokens / 1_000_000) * OUTPUT_PRICE_PER_MILLION
    total_cost = input_cost + output_cost

    return {
        "input_cost": input_cost,
        "output_cost": output_cost,
        "total_cost": total_cost,
    }

result = calculate_gpt54mini_price(652, 33)
print("Total cost: $", round(result["total_cost"], 8))

Total cost: $ 0.0001176


In [65]:
instructions = """
You're a course teaching assistant.
You're given a question from a course student and your task is to answer it.

If you want to look up information, use the search function. 
Use as many keywords from the user question as possible when making first requests.

Make multiple searches.

Try to expand your search by using new keywords
based on the results you get from the search.

At the end, ask if there are other areas that the user wants to explore.
""".strip()

In [66]:
def make_call(call):
    args = json.loads(call.arguments)

    result = ''
    if call.name == "search":
        result = search(**args)
    
    result_json = json.dumps(result, indent=2)

    return {
        "type": "function_call_output",
        "call_id": call.call_id,
        "output": result_json
    }

In [67]:
question = "I just discovered the course. Can I join it?"

messages = [
    {"role": "developer", "content": instructions},
    {"role":"user", "content": question}
]

response = openai_client.responses.create(
    model = "gpt-5.4-mini",
    input = messages,
    tools = [search_tool]
)

messages.extend(response.output)
has_function_calls = False

for item in response.output:
    if item.type == "function_call":
        print("function_call:", item.name, item.arguments)
        call_output = make_call(item)
        messages.append(call_output)
        has_function_calls = True
    
    elif item.type == "message":
        print("ASSISTANT:")
        print(item.content[0].text)

function_call: search {"query":"join course discovered the course can I join enrollment registration access FAQ"}


In [77]:
def agent_loop(instructions, question, model="gpt-5.4-mini") -> str:


    messages = [
        {"role": "developer", "content": instructions},
        {"role":"user", "content": question}
    ]

    last_answer = ''

    it = 1

    while True:
        print(f"iteration #{it}...")
        has_function_calls = False

        response = openai_client.responses.create(
            model = model,
            input = messages,
            tools = [search_tool]
        )

        usage = response.usage
        print(f'Tokens({usage.input_tokens}, {usage.output_tokens})')
        result = calculate_gpt54mini_price(usage.input_tokens, usage.output_tokens)
        print("Total cost: $", round(result["total_cost"], 8))

        messages.extend(response.output)

        for item in response.output:
            if item.type == "function_call":
                print("function_call:", item.name, item.arguments)
                call_output = make_call(item)
                messages.append(call_output)
                has_function_calls = True
            
            elif item.type == "message":
                last_answer = item.content[0].text
        
        it = it + 1
        if has_function_calls == False:
            break
    return last_answer

In [79]:
instructions = """
You're a course teaching assistant.
You're given a question from a course student and your task is to answer it.

If you want to look up information, use the search function. 
Use as many keywords from the user question as possible when making first requests.

Make multiple searches. First perform search, analyze the results 
and then perform more searches. 

The question has to be about the course or its logistics, offtopic questions 
shouldn't be answered. If the search returns nothing, it's likely an off-topic question.
If you can't answer the question using FAQ, don't do it yourself. Only use the 
facts from the FAQ database.

At the end, ask if there are other areas that the user wants to explore.
""".strip()

In [80]:
question = "What's qeen gambit"

answer = agent_loop(instructions, question)
print(answer)




iteration #1...


Tokens(213, 22)
Total cost: $ 4.515e-05
function_call: search {"query":"queen's gambit chess opening"}
iteration #2...
Tokens(382, 22)
Total cost: $ 7.05e-05
function_call: search {"query":"what is qeen gambit"}
iteration #3...
Tokens(1299, 65)
Total cost: $ 0.00023385
It looks like this isn’t a course-related question, and I couldn’t find a matching FAQ entry for it.

If you meant the chess opening, it’s spelled **“Queen’s Gambit.”** If you want, I can help with course-related questions instead—anything else you want to explore?


In [81]:
from toyaikit.llm import OpenAIClient
from toyaikit.tools import Tools
from toyaikit.chat import IPythonChatInterface
from toyaikit.chat.runners import OpenAIResponsesRunner, DisplayingRunnerCallback

In [84]:
def search(query: str) -> dict[str,str]:
    """
    Search the FAQ database for entries matching the given query.
    """

    return index.search(
        query,
        num_results=5,
        boost_dict={'question':3.0, 'section':0.5},
        filter_dict={'course':'llm-zoomcamp'}
    )


In [89]:
agent_tools = Tools()
agent_tools.add_tool(search)

In [90]:
agent_tools.get_tools()

[{'type': 'function',
  'name': 'search',
  'description': 'Search the FAQ database for entries matching the given query.',
  'parameters': {'type': 'object',
   'properties': {'query': {'type': 'string',
     'description': 'query parameter'}},
   'required': ['query'],
   'additionalProperties': False}}]

In [93]:
chat_interface = IPythonChatInterface()
callback = DisplayingRunnerCallback(chat_interface)

In [92]:
runner = OpenAIResponsesRunner(
    tools=agent_tools,
    developer_prompt=instructions,
    chat_interface=chat_interface,
    llm_client=OpenAIClient(model='gpt-5.4-mini')
)

In [95]:
result = runner.loop(
    prompt='How do i run Olamo locally?',
    callback=callback
)

-> Response received


-> Response received


-> Response received


In [96]:
result.cost

CostInfo(input_cost=Decimal('0.00239775'), output_cost=Decimal('0.001305'), total_cost=Decimal('0.00370275'))

In [98]:
result = runner.loop(
    prompt='How do i run a different model?',
    previous_messages=result.all_messages,
    callback=callback
)

-> Response received


-> Response received


In [99]:
runner.run()

-> Response received


-> Response received


-> Response received


-> Response received


-> Response received


-> Response received


-> Response received


Chat ended.


LoopResult(new_messages=[EasyInputMessage(content="You're a course teaching assistant.\nYou're given a question from a course student and your task is to answer it.\n\nIf you want to look up information, use the search function. \nUse as many keywords from the user question as possible when making first requests.\n\nMake multiple searches. First perform search, analyze the results \nand then perform more searches. \n\nThe question has to be about the course or its logistics, offtopic questions \nshouldn't be answered. If the search returns nothing, it's likely an off-topic question.\nIf you can't answer the question using FAQ, don't do it yourself. Only use the \nfacts from the FAQ database.\n\nAt the end, ask if there are other areas that the user wants to explore.", role='developer', phase=None, type=None), EasyInputMessage(content='naku', role='user', phase=None, type=None), ResponseFunctionToolCall(arguments='{"query":"naku"}', call_id='call_C3sYorRX04rfBoDnjzFh7ffL', name='search'